## 20 Complex SQL Questions

**SaaS Subscription Analytics Star Schema**

**Database: saas_subscription_dw.db**

**Schema docs: see SCHEMA_NOTES.md**

In [1]:
# -- SETUP" Run this cell first before attempting questions
import sqlite3
import pandas as pd

DB_PATH = r'/Users/ajwarck/Desktop/vs_code/sql/practice_dir/saas_subscription/database/saas_subscription_dw.db'
conn = sqlite3.connect(DB_PATH)

# Load all tables into DataFrame upfront - ready to use
dim_date = pd.read_sql_query('select * from dim_date', conn)
dim_plan = pd.read_sql_query('select * from dim_plan', conn)
dim_sales_rep = pd.read_sql_query('select * from dim_sales_rep', conn)
dim_customer = pd.read_sql_query('select * from dim_customer', conn)
fact_subscription_events = pd.read_sql_query('select * from fact_subscription_events', conn)
fact_daily_subscription_snapshot = pd.read_sql_query('select * from fact_daily_subscription_snapshot', conn)
fact_invoice_payments = pd.read_sql_query('select * from fact_invoice_payments', conn)
fact_product_usage = pd.read_sql_query('select * from fact_product_usage', conn)

print("Setup complete! All tables loaded.")
print("Dimension tables are ready: dim_date, dim_plan, dim_sales_rep, dim_customer")
print("Fact tables are ready: fact_subscription_events, fact_daily_subscription_snapshot, fact_invoice_payments, fact_product_usage")

def run(query):
    """Helper function - pass any SQL query and get a DataFrame back"""
    return pd.read_sql_query(query, conn)

print('\nConnected database Successfully!!!')

Setup complete! All tables loaded.
Dimension tables are ready: dim_date, dim_plan, dim_sales_rep, dim_customer
Fact tables are ready: fact_subscription_events, fact_daily_subscription_snapshot, fact_invoice_payments, fact_product_usage

Connected database Successfully!!!


In [3]:
print('=' * 75)
print('Dimension Tables Sample')
print('=' * 75)
print('1. date table:\n', dim_date.head(2))
print('\n2. plan table:\n', dim_plan.head(2))
print('\n3. sales_rep table:\n', dim_sales_rep.head(2))
print('\n4. customer table:\n', dim_customer.head(2))

Dimension Tables Sample
1. date table:
      date_key  year  quarter  month month_name  day_of_month  day_of_week  \
0  2025-01-01  2025        1      1    January             1            2   
1  2025-01-02  2025        1      1    January             2            3   

   is_weekend  is_month_end fiscal_period  
0           0             0       2025-01  
1           0             0       2025-01  

2. plan table:
    plan_key  plan_code        plan_name  plan_tier_rank billing_period  \
0         1  STARTER_M  Starter Monthly               1        monthly   
1         2  STARTER_A   Starter Annual               1         annual   

   list_price_per_seat currency  
0                 29.0      USD  
1                 24.0      USD  

3. sales_rep table:
    rep_key     rep_name team region
0        1  Lena Brooks  SMB   AMER
1        2  Marcus Webb  SMB   EMEA

4. customer table:
    customer_sk  customer_id         customer_name    industry segment country  \
0            1        

In [4]:
print('=' * 75)
print('Fact Tables Sample')
print('=' * 75)
print('1. subscription events table:\n', fact_subscription_events.head(2))
print('\n2. daily subscription snapshot table:\n', fact_daily_subscription_snapshot.head(2))
print('\n3. invoice payments table:\n', fact_invoice_payments.head(2))
print('\n4. product usage table:\n', fact_product_usage.head(2))

Fact Tables Sample
1. subscription events table:
    event_id  subscription_id  customer_id  event_date       event_type  \
0         1               10            1  2025-01-06      trial_start   
1         2               10            1  2025-01-20  trial_converted   

   plan_key  prior_plan_key  seats  mrr_impact cancel_reason  
0         1             NaN      3         0.0          None  
1         1             1.0      3        87.0          None  

2. daily subscription snapshot table:
   snapshot_date  subscription_id  customer_id  plan_key  seats  mrr    status
0    2025-01-06               10            1         1      3  0.0  trialing
1    2025-01-07               10            1         1      3  0.0  trialing

3. invoice payments table:
    invoice_id  subscription_id  customer_id invoice_date  amount_due  \
0           1               10            1   2025-01-20        87.0   
1           2               10            1   2025-02-19        87.0   

   amount_paid pay

### **Q1. MRR Waterfall by month.**
**Business Question:** "Give me the standard MRR bridge (new, expansion, contraction, churn, reactivation) by month."

This is the core SaaS finance report - every subscription company's board deck has this exact table.

In [22]:
query = """
    with monthly as (
        select 
            strftime('%Y-%m', event_date) as period,
            sum(case when event_type='trial_converted' then mrr_impact else 0 end) as new_mrr,
            sum(case when event_type='upgrade' and mrr_impact>0 then mrr_impact else 0 end) as expansion_mrr,
            sum(case when event_type='downgrade' then mrr_impact else 0 end) as contraction_mrr,
            sum(case when event_type='cancelled' then mrr_impact else 0 end) as churned_mrr,
            sum(case when event_type='reactivated' then mrr_impact else 0 end) as reactivation_mrr 
        from fact_subscription_events
        group by 1
    )
    select 
        period,
        round(new_mrr, 2)           as new_mrr,
        round(expansion_mrr, 2)     as expansion_mrr,
        round(contraction_mrr, 2)   as contraction_mrr,
        round(churned_mrr, 2)       as churned_mrr,
        round(reactivation_mrr, 2)  as reactivation_mrr,
        round(new_mrr + expansion_mrr + contraction_mrr + churned_mrr + reactivation_mrr, 2) as net_new_mrr
    from monthly
    order by period;
"""
run(query)

,period,new_mrr,expansion_mrr,contraction_mrr,churned_mrr,reactivation_mrr,net_new_mrr
0,2025-01,864.0,0.0,0.0,0.0,0.0,864.0
1,2025-02,2276.0,2042.0,0.0,0.0,0.0,4318.0
2,2025-03,3158.0,4448.0,-2618.0,-232.0,0.0,4756.0
3,2025-04,2589.0,208.0,-308.0,-719.0,0.0,1770.0
4,2025-05,3740.0,3482.0,0.0,-232.0,0.0,6990.0
5,2025-06,174.0,882.0,-1029.0,-980.0,0.0,-953.0
6,2025-07,0.0,287.0,0.0,-406.0,87.0,-32.0
7,2025-08,0.0,0.0,0.0,-232.0,0.0,-232.0


### **Q2. Ending MRR / ARR and Active Subscription COUNT By Month-end**
**Business Question:** "What was our MRR/ARR at the end of every month?"

Demonstrates the snapshot-fact pattern: instead of replying events, read the state directly off fact_daily_subscription_snapshot.

In [64]:
query = """
    with month_ends as (
        select 
            fiscal_period,
            max(date_key) as month_end_date
        from dim_date
        where date_key <= (select max(snapshot_date) from fact_daily_subscription_snapshot)
        group by fiscal_period
    )
    select 
        me.fiscal_period,
        round(sum(s.mrr), 2) as ending_mrr,
        round(sum(s.mrr) * 12, 2) as ending_arr,
        count(distinct s.subscription_id) as active_subscriptions
    from month_ends as me
    join fact_daily_subscription_snapshot as s on me.month_end_date = s.snapshot_date
                                                and s.status = 'active'
    group by me.fiscal_period
    order by me.fiscal_period
"""
run(query)

,fiscal_period,ending_mrr,ending_arr,active_subscriptions
0,2025-01,864.0,10368.0,3
1,2025-02,5182.0,62184.0,8
2,2025-03,9938.0,119256.0,13
3,2025-04,11476.0,137712.0,16
4,2025-05,18698.0,224376.0,22
5,2025-06,17513.0,210156.0,20
6,2025-07,17626.0,211512.0,18
7,2025-08,17394.0,208728.0,17
8,2025-09,17394.0,208728.0,17


### **Q3. LOGO Churn Rate & Revenue Churn Rate, Month Over Month**
**Business Question:** "What % of customers/revenue churned each month, relative to who was active at the start of that month?"

Uses LAG-style prior-month join on the snapshot fact (not events) so the denominator is exactly "active at start of period" - the textbook-correct churn rate definition that's easy to get wrong.

In [138]:
query = """
    with month_ends as (
        select 
            fiscal_period,
            max(date_key) as month_end_date 
        from dim_date
        where date_key <= (select max(snapshot_date) from fact_daily_subscription_snapshot)
        group by fiscal_period
    ),
    ordered_month as (
        select 
            fiscal_period,
            month_end_date,
            lag(month_end_date) over(order by fiscal_period) as prev_month_end
        from month_ends
    ),
    month_snapshot as (
        select 
            om.fiscal_period,
            om.month_end_date,
            om.prev_month_end,
            s.subscription_id,
            s.mrr,
            s.status 
        from ordered_month as om
        join fact_daily_subscription_snapshot as s on om.month_end_date = s.snapshot_date
    ),
    prev_snapshot as (
        select 
            om.fiscal_period,
            om.prev_month_end,
            s.subscription_id,
            s.mrr as prev_mrr,
            s.status as prev_status
        from ordered_month as om
        join fact_daily_subscription_snapshot as s on om.prev_month_end = s.snapshot_date
    )
    select 
        ms.fiscal_period,
        count(distinct case when ps.prev_status='active' then ps.subscription_id end) as starting_active_logos,
        count(distinct case when ps.prev_status='active' and ms.status='cancelled' then ps.subscription_id end) as churned_logos,
        round(
            count(distinct case when ps.prev_status='active' and ms.status='cancelled' then ps.subscription_id end) * 1.0 /
            count(distinct case when ps.prev_status='active' then ps.subscription_id end), 4
        ) as churned_rate,
        sum(case when ps.prev_status='active' then ps.prev_mrr else 0 end) as starting_mrr,
        sum(case when ps.prev_status='active' and ms.status='cancelled' then ps.prev_mrr else 0 end) as churned_mrr,
        round(
            sum(case when ps.prev_status='active' and ms.status='cancelled' then ps.prev_mrr else 0 end) * 1.0 /
            sum(case when ps.prev_status='active' then ps.prev_mrr else 0 end), 4
        ) as revenue_churn_rate
    from month_snapshot as ms 
    join prev_snapshot as ps on ms.fiscal_period = ps.fiscal_period
                            and ms.subscription_id = ps.subscription_id
    group by ms.fiscal_period
    order by ms.fiscal_period
"""
run(query)

,fiscal_period,starting_active_logos,churned_logos,churned_rate,starting_mrr,churned_mrr,revenue_churn_rate
0,2025-02,3,0,0.0000,864.0,0.0,0.0000
1,2025-03,7,0,0.0000,4950.0,0.0,0.0000
2,2025-04,12,1,0.0833,9851.0,632.0,0.0642
3,2025-05,16,0,0.0000,11476.0,0.0,0.0000
4,2025-06,19,0,0.0000,17718.0,0.0,0.0000
5,2025-07,18,0,0.0000,17339.0,0.0,0.0000
6,2025-08,17,0,0.0000,17394.0,0.0,0.0000
7,2025-09,17,0,0.0000,17394.0,0.0,0.0000
